# Week 7: Planning and Learning Integration

Craig Rudman  
Regis University  
MSDS684 Reinforcement Learning  
Prof. Mike Busch  

## Section 1: Project Overview 

This lab examines how model-learning changes the sample efficiency of tabular Q-learning methods. The lab also investigates how to detect when a model needs updating due to changes in the environment, and how stale entries are overwritten in such cases. The lab will help explain under what conditions a model adds value and what mechanism delivers it.

I explore the concept of *planning*, which Sutton and Barto define as “any computational process that takes a model as input and produces or improves a policy for interacting with the modeled environment” (Sutton and Barto, p. 160). I will demonstrate how planning and Q-learning are integrated with the Dyna architecture, and compare the ability of Dyna-Q and Dyna-Q+ to respond to environment changes. I will look at prioritized sweeping to improve planning efficiency. Finally, I will examine the computational trade-offs involved when using model-based versus model-free methods.

The lab uses Gymnasium's Taxi-v4 in deterministic mode to conduct this investigation, since Taxi-v3 has been deprecated. This release has the following attributes:

- 500 discrete states (25 taxi positions in a 5x5 grid, 5 possible passenger locations, and 4 destination locations)
- 6 discrete actions (up, down, left, right, pickup, dropoff)
- Rewards per step: +20 for delivering a passenger, -10 for incorrect pickup or dropoff, and -1 for all other steps
- Episodes terminate when a taxi drops off a passenger and terminate after 200 steps when the TimeLimit wrapper is used to construct the environment

I have the following expectations from this lab:

- Dyna-Q will accumulate rewards faster, require fewer steps per episode, and fewer episodes overall than Q-learning alone to achieve optimal performance. Q-learning alone makes one policy update per step; with planning, multiple Q-updates are made per real step depending on the number of planning steps configured. Sutton and Barto demonstrate this on the maze task: without planning, the policy extends backward one step per episode; with planning, it extends much further per episode (Sutton and Barto, p. 165).

- Dyna-Q+ will adapt to environment changes in fewer steps than Dyna-Q. Dyna-Q+ adds an exploration bonus based on the last time a state-action pair was tried in the environment, increasing the chance that stale transitions will be revisited.

- Prioritized sweeping will converge on an optimal solution with fewer updates than uniform random planning. It maintains a reverse index of the model that maps states to their state-action predecessors, and a priority queue ranked by absolute TD error. These mechanisms direct the planning loop to propagate updates backwards to state-action predecessors whose Q-values are most likely to change, rather than sampling them uniformly.

## Section 2: Deliverables

### GitHub Repository

GitHub Repository: https://github.com/craig-rudman/MSDS684/tree/main/W7

### Implementation Summary

In [1]:
'''Implementation Summary (100-150 words)

Brief prose description:

What you implemented (algorithms, environments)
Experimental setup (e.g., “1000 episodes, 30 random seeds, ε=0.1”)
Key hyperparameters chosen
NOT detailed pseudocode or line-by-line methods
'''


'Implementation Summary (100-150 words)\n\nBrief prose description:\n\nWhat you implemented (algorithms, environments)\nExperimental setup (e.g., “1000 episodes, 30 random seeds, ε=0.1”)\nKey hyperparameters chosen\nNOT detailed pseudocode or line-by-line methods\n'

#### Key Results and Analysis

In [2]:
'''Key Results & Analysis (400-600 words + 2-4 visualizations)
⚠️ CRITICAL RULES:

NO raw code listings in PDF
NO console output dumps
Code lives in GitHub; analysis lives here
Include 2-4 key visualizations:

Embedded directly in PDF (PNG/JPG format)
Each must have detailed interpretive caption
Choose your most important/insightful results
Captions must be interpretive, not just descriptive:

❌ Bad caption: “Figure 1 shows a line going up”

✅ Good caption: “Figure 1: UCB’s regret curve flattens after 500 steps because the
uncertainty bonus decreases as actions are sampled more, focusing exploration on 
genuinely uncertain actions (S&B Ch. 2.7). 
Final regret is 15% lower than ε-greedy (ε=0.1), demonstrating more efficient exploration.”

Discussion must address:

What do results show about algorithm behavior?
How do they relate to theory from Sutton & Barto? (cite chapters/sections)
What didn’t work as expected? Why?
How did hyperparameters affect performance?
What does this teach you about the RL concept?
Focus: Interpretation and understanding, not exhaustive description of every result.
'''

'Key Results & Analysis (400-600 words + 2-4 visualizations)\n⚠️ CRITICAL RULES:\n\nNO raw code listings in PDF\nNO console output dumps\nCode lives in GitHub; analysis lives here\nInclude 2-4 key visualizations:\n\nEmbedded directly in PDF (PNG/JPG format)\nEach must have detailed interpretive caption\nChoose your most important/insightful results\nCaptions must be interpretive, not just descriptive:\n\n❌ Bad caption: “Figure 1 shows a line going up”\n\n✅ Good caption: “Figure 1: UCB’s regret curve flattens after 500 steps because the\nuncertainty bonus decreases as actions are sampled more, focusing exploration on \ngenuinely uncertain actions (S&B Ch. 2.7). \nFinal regret is 15% lower than ε-greedy (ε=0.1), demonstrating more efficient exploration.”\n\nDiscussion must address:\n\nWhat do results show about algorithm behavior?\nHow do they relate to theory from Sutton & Barto? (cite chapters/sections)\nWhat didn’t work as expected? Why?\nHow did hyperparameters affect performance?\nW

## Section 3: AI Use Reflection

### 3.1 Initial Interaction

I began by specifying roles, asking Claude to act as tutor and critic; I would design the implementation and draft the report. Our first task was to clarify the distinction between model-free and model-based RL, which Claude probed with a thought experiment about what happens after one real step with 50 planning iterations. This discussion ensured that I understood the role the model plays in developing policy and supported the Section One draft.

### 3.2 Iteration Cycle

#### 3.2.1 Iteration Cycle - Example #1

In drafting the project overview, I wrote that "Q-planning will develop policies multiple times per step." Claude challenged the phrasing on two counts: planning doesn't develop policy, and Q-planning isn't standard terminology.  I shared a citation from the textbook that supported the role planning plays in developing policy. Claude conceded that it was valid, so I kept the policy-development framing, supported by the citation, but accepted the terminology critique by dropping 'Q-planning.'

#### 3.2.2 Iteration Cycle - Example #2

A lengthy exchange with Claude led me to understand that there are four key data structures that enable model-based RL with prioritized sweeping: the Q-table, model, reverse index, and priority queue. Along the way I corrected several misconceptions: I had typed the queue as a stack, and I thought it was ordered by proximity rather than absolute TD error. I synthesized the four-structure summary myself.

#### 3.2.3 Iteration Cycle - Example #3

When I was reviewing Gymnasium documentation, I saw that Taxi-v3 had been deprecated. I proceeded to plan the implementation around Taxi-v4. When Claude ran the first iteration's tests, it reported a VersionNotFound error because my conda environment was configured with an earlier version of gymnasium. I upgraded gymnasium and our tests went green.

### 3.3 Critical Evaluation

Iterative development with a steel-thread proof-of-concept exposed an early Claude over-engineering: the per-step trace schema produced 12 MB files after one short Q-learning run. I proposed per-episode aggregation as an alternative, which preserved every metric the visualizations needed at ~370 KB per run. Test-first refactoring and re-running the cumulative-reward demo confirmed the new schema produced the same learning curve before we committed.

### 3.4 Learning Reflection

In [4]:
'''
(50-75 words)

What did you learn about the RL algorithm through debugging?
What did you learn about working with AI tools?
What would you do differently next time?
'''

'\n(50-75 words)\n\nWhat did you learn about the RL algorithm through debugging?\nWhat did you learn about working with AI tools?\nWhat would you do differently next time?\n'

In [5]:
'''
Grading Criteria

Excellent (30-35 points):
    Documents 3+ specific debugging cycles
    Shows critical evaluation of AI suggestions
    Demonstrates learning from the process
    Specific, detailed examples

Good (24-29 points):
    Documents 2 debugging cycles
    Some critical thinking shown
    Generic learning statements
    Less specific

Adequate (18-23 points):
    Mentions iteration but lacks detail
    Minimal critical evaluation
    Vague descriptions

Poor (0-17 points):
    “I used AI to generate code”
    No iteration documented
    No evidence of critical thinking
    Appears to accept first draft
'''

'\nGrading Criteria\n\nExcellent (30-35 points):\n    Documents 3+ specific debugging cycles\n    Shows critical evaluation of AI suggestions\n    Demonstrates learning from the process\n    Specific, detailed examples\n\nGood (24-29 points):\n    Documents 2 debugging cycles\n    Some critical thinking shown\n    Generic learning statements\n    Less specific\n\nAdequate (18-23 points):\n    Mentions iteration but lacks detail\n    Minimal critical evaluation\n    Vague descriptions\n\nPoor (0-17 points):\n    “I used AI to generate code”\n    No iteration documented\n    No evidence of critical thinking\n    Appears to accept first draft\n'

## Section 4: Speaker Notes

In [6]:
'''
(~5 minutes / 5-7 bullets)

Weight: 10 points (10%)
Purpose: Practice communicating technical work concisely. These could be used for peer presentations or your own reference.

Outline a brief presentation of your work:

Format: 5-7 bullet points covering:

Problem statement and motivation
Method and key algorithmic choices
Important design decision or challenge you faced
Main result or finding
Key insight or learning
(Optional) Connection to future weeks or real-world applications
Example:

• Problem: Exploration-exploitation in 10-armed bandits with non-stationary rewards
• Method: Implemented ε-greedy and UCB with constant step-size (α=0.1)
• Design choice: Used constant α instead of sample average to track changing rewards
• Key result: UCB outperformed ε-greedy after 500 steps (15% lower regret)
• Insight: Uncertainty-based exploration more efficient than random
• Challenge: Debugging action selection logic took 6 iterations with Claude
• Connection: This exploration concept extends to full MDPs in Week 2
'''

'\n(~5 minutes / 5-7 bullets)\n\nWeight: 10 points (10%)\nPurpose: Practice communicating technical work concisely. These could be used for peer presentations or your own reference.\n\nOutline a brief presentation of your work:\n\nFormat: 5-7 bullet points covering:\n\nProblem statement and motivation\nMethod and key algorithmic choices\nImportant design decision or challenge you faced\nMain result or finding\nKey insight or learning\n(Optional) Connection to future weeks or real-world applications\nExample:\n\n• Problem: Exploration-exploitation in 10-armed bandits with non-stationary rewards\n• Method: Implemented ε-greedy and UCB with constant step-size (α=0.1)\n• Design choice: Used constant α instead of sample average to track changing rewards\n• Key result: UCB outperformed ε-greedy after 500 steps (15% lower regret)\n• Insight: Uncertainty-based exploration more efficient than random\n• Challenge: Debugging action selection logic took 6 iterations with Claude\n• Connection: This